# RealityCheck GenImage v2 — guarded Kaggle training

This notebook keeps the frozen v1 model intact, recreates its exact SID_Set sample, prepares a deterministic balanced subset of **Unbiased Tiny GenImage**, warm-starts `model_v2.safetensors`, evaluates v1 and v2 across all 20 published scenarios, and creates `genimage_v2_export.zip`.

Before choosing **Run All**: enable a **T4 GPU** and **Internet**, then choose **Add Input** and attach `cartografia/unbiased-tiny-genimage` version 1. The participant confirmed permission to use the GenImage/ImageNet-derived data for this event on 2026-08-31. WildFake is never read by this workflow.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import torch

REPO_URL = "https://github.com/LINGSIHAN/TikTok-Hackathon-Track-5.git"
BRANCH = "master"
PROJECT_DIR = Path("/kaggle/working/TikTok-Hackathon-Track-5")
DATASET_ROOT = Path("/kaggle/input/unbiased-tiny-genimage")
LICENSE_CONFIRMED = True

def run(*args):
    command = [str(value) for value in args]
    print("+", " ".join(command), flush=True)
    subprocess.run(command, check=True)

if not LICENSE_CONFIRMED:
    raise RuntimeError("Dataset permission must be confirmed before training.")
if not DATASET_ROOT.is_dir():
    raise RuntimeError(
        "Missing /kaggle/input/unbiased-tiny-genimage. Use Add Input and "
        "attach cartografia/unbiased-tiny-genimage version 1."
    )
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Choose a T4 GPU in Kaggle Settings and restart.")
GPU_NAME = torch.cuda.get_device_name(0)
if "T4" not in GPU_NAME.upper():
    raise RuntimeError(f"This locked workflow requires a T4 GPU; found {GPU_NAME}.")
print("GPU:", GPU_NAME)
print("Attached dataset:", DATASET_ROOT)

In [ ]:
if PROJECT_DIR.exists():
    if not (PROJECT_DIR / ".git").is_dir():
        raise RuntimeError(f"{PROJECT_DIR} exists but is not the expected Git checkout.")
    origin = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "remote", "get-url", "origin"],
        text=True,
    ).strip()
    if origin.rstrip("/").removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError(f"Unexpected existing repository origin: {origin}")
    tracked = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"],
        text=True,
    ).strip()
    if tracked:
        raise RuntimeError("Tracked changes exist in the Kaggle checkout; preserve outputs and start a fresh session.")
    run("git", "-C", PROJECT_DIR, "fetch", "--depth", "50", "origin", BRANCH)
    run("git", "-C", PROJECT_DIR, "merge", "--ff-only", "FETCH_HEAD")
else:
    run("git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, PROJECT_DIR)

os.chdir(PROJECT_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository commit:", COMMIT)
run(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt")

In [ ]:
run(
    sys.executable,
    "scripts/run_genimage_v2_kaggle.py",
    "--input-root", DATASET_ROOT,
    "--output-root", "/kaggle/working",
    "--license-confirmed",
)

In [ ]:
archive = Path("/kaggle/working/genimage_v2_export.zip")
if not archive.is_file() or archive.stat().st_size == 0:
    raise RuntimeError("The validated export was not created. Review the first failed cell.")
print("Download this file from Kaggle Output:", archive)